[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/A.%20Foundational%20Analytics%20%26%20Inventory%20Control%20%28The%20Language%20of%20Supply%20Chain%29/056.%20The%20Safety%20Stock%20%26%20Reorder%20Point%20%28Q%2Cr%29%20Policy%20Problem/P56-Tier-2_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 56. The Safety Stock and Reorder Point \((Q,r)\) Policy Problem

## Tier 2: Priority-Based Heuristic Algorithm

### Purpose and scope

This notebook applies a priority-based heuristic to an inventory problem. It computes an order quantity \(Q\), a reorder point \(r\), and safety stock for each of three candidate strategies. A priority queue determines the order in which the strategies are evaluated; the candidate with the smallest estimated annual cost is then selected.

The notebook also compares the selected candidate with a mathematical-formulation baseline, visualizes the calculations, and applies the same procedure to several product examples. The heuristic is a candidate-selection procedure, not a proof of global optimality. The cost calculation includes a simplified stockout-cost proxy, so its output should be interpreted as an estimate under the assumptions below rather than as a general-purpose inventory cost forecast.

### Background and notation

In a continuous-review \((Q,r)\) policy, an order of size \(Q\) is placed when the inventory position reaches the reorder point \(r\). Safety stock protects against uncertainty in demand during lead time. The model uses:

- \(D\): annual demand, in units per year.
- \(\mu=D/365\): mean daily demand, in units per day.
- \(L\) and \(\sigma_L\): mean and standard deviation of lead time, in days.
- \(\sigma_D\): standard deviation of daily demand, in units per day.
- \(h\): annual holding cost per unit, calculated as the holding-cost rate times unit cost.
- \(K\): ordering cost per order, and \(c_s\): stockout cost per unit in the simplified cost proxy.
- \(z\): the standard normal quantile corresponding to the target service level.

The safety-stock formula assumes that daily demand and lead-time uncertainty are independent and that the variance of demand accumulated over a random lead time can be approximated by the sum of the demand-variance and lead-time-variance terms. The EOQ calculation uses annual demand and annual holding cost, so its units are consistent. Demand standard deviation is interpreted as a daily quantity throughout.

### Priority score and candidate strategies

The criticality score is a heuristic index formed from the stockout-to-holding cost ratio and the coefficients of variation. It is used to assign queue priority; it is not itself an objective function or a calibrated probability of risk. The queue ranks the three strategy candidates, but every candidate is evaluated before the lowest estimated-cost candidate is selected.

The candidates are cost-focused, service-focused, and balanced. They share the same EOQ calculation. Their safety-stock rules differ: the cost-focused candidate scales the standard safety stock according to the cost ratio, the service-focused candidate adds a buffer when the target service level exceeds 0.99, and the balanced candidate uses the standard safety-stock calculation without adjustment.

### Mathematical model used by the calculations

The combined standard deviation of demand over lead time is approximated by

$$
\sigma_{DL}=\sqrt{L\sigma_D^2+\mu^2\sigma_L^2}.
$$

For target service level \(\alpha\), the notebook calculates safety stock, EOQ, and reorder point as

$$
\begin{aligned}
z &= \Phi^{-1}(\alpha), \\
SS &= \max(0,z\sigma_{DL}), \\
Q^* &= \sqrt{\frac{2DK}{h}}, \\
r &= \mu L+SS.
\end{aligned}
$$

Here, \(\Phi^{-1}\) is the inverse standard normal cumulative distribution function. The lower bound of zero on safety stock prevents a negative buffer when the selected service-level quantile is negative.

For a candidate \((Q,r,SS)\), the estimated annual cost is

$$
\widehat{C}(Q,SS)=h\left(\frac{Q}{2}+SS\right)+K\frac{D}{Q}
+c_s\left((1-\alpha)\frac{D}{4}\right).
$$

The first term estimates annual holding cost for cycle stock and safety stock; the second estimates annual ordering cost. The final term is a simplified stockout-cost proxy: it treats the fraction \(1-\alpha\) of annual demand as exposed over a quarter-year. It is not derived from a lead-time demand distribution or from an explicit expected-shortage calculation. Because this proxy is constant across the candidates for a given item, the candidate comparison is driven by the holding-cost differences; it should not be interpreted as a detailed service-cost optimization.

The selected candidate is the evaluated strategy with the smallest \(\widehat{C}\). If two candidates have equal estimated costs, the first encountered in priority-queue order remains selected. The queue uses a min-heap with negated priority values so that higher numerical priorities are processed first.

### Algorithm and computational effort

For each item, the algorithm builds a heap containing three strategies, repeatedly removes the highest-priority strategy, calculates its candidate values, evaluates estimated cost, and retains the least-cost candidate. With \(s\) strategies, heap construction and removals take \(O(s\log s)\) time in total, and the heap uses \(O(s)\) space. Here, \(s=3\); the calculation for each candidate takes constant time. For multiple products, the same procedure is run independently for each product.

### Modeling assumptions and interpretation

- Demand and lead-time inputs are treated as daily demand statistics and day-based lead-time statistics, respectively; a year is taken to have 365 days.
- The standard safety-stock calculation uses the stated normal-quantile approximation. The strategy adjustments are heuristic rules, not additional probabilistic guarantees.
- The cost proxy is intended for comparing the candidates within this example. It does not explicitly model backorders, lost sales, replenishment dynamics, or the expected quantity of each shortage.
- The priority score and the three candidate rules are illustrative decision aids. They do not establish that the selected candidate is globally optimal.

### What to examine in the results

Inspect how the cost ratio and variability coefficients contribute to the criticality index, how queue priority determines evaluation order, and how the safety-stock rules affect estimated holding cost. Compare the heuristic candidate with the mathematical baseline while keeping in mind that the baseline is evaluated using the same simplified cost calculation.

In [ ]:
# Load the numerical, tabular, plotting, and priority-queue tools used in this notebook.
import heapq
import math
from dataclasses import dataclass
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully for the priority-based heuristic.")

In [ ]:
@dataclass
class InventoryParameters:
    """Input data and derived daily and annual inventory quantities."""

    annual_demand: float
    lead_time_mean: float
    lead_time_std: float
    demand_std: float
    holding_cost_rate: float
    unit_cost: float
    ordering_cost: float
    stockout_cost: float
    service_level: float

    def __post_init__(self):
        if self.annual_demand <= 0:
            raise ValueError("Annual demand must be positive.")
        if self.lead_time_mean <= 0:
            raise ValueError("Mean lead time must be positive.")
        if self.lead_time_std < 0 or self.demand_std < 0:
            raise ValueError("Standard deviations cannot be negative.")
        if self.holding_cost_rate <= 0 or self.unit_cost <= 0:
            raise ValueError("Holding-cost rate and unit cost must be positive.")
        if self.ordering_cost <= 0:
            raise ValueError("Ordering cost must be positive.")
        if self.stockout_cost < 0:
            raise ValueError("Stockout cost cannot be negative.")
        if not 0 < self.service_level < 1:
            raise ValueError("Service level must be strictly between 0 and 1.")

        self.daily_demand = self.annual_demand / 365
        self.holding_cost = self.holding_cost_rate * self.unit_cost
        self.z_score = stats.norm.ppf(self.service_level)


class PriorityBasedQRPolicy:
    """Calculate the criticality index and standard (Q, r) policy quantities."""

    def __init__(self, params: InventoryParameters):
        self.params = params
        self.criticality_score = self.calculate_criticality_score()

    def calculate_criticality_score(self) -> float:
        """Return the heuristic score used to rank strategy candidates."""
        cost_ratio = self.params.stockout_cost / self.params.holding_cost
        demand_cv = self.params.demand_std / self.params.daily_demand
        lead_time_cv = self.params.lead_time_std / self.params.lead_time_mean

        return cost_ratio * (1 + demand_cv) * (1 + lead_time_cv)

    def calculate_safety_stock(self) -> float:
        """Calculate safety stock from demand and lead-time variability."""
        demand_variance = (
            self.params.lead_time_mean * self.params.demand_std ** 2
        )
        lead_time_variance = (
            self.params.daily_demand ** 2 * self.params.lead_time_std ** 2
        )
        combined_std = math.sqrt(demand_variance + lead_time_variance)
        safety_stock = self.params.z_score * combined_std

        return max(0.0, safety_stock)

    def calculate_eoq(self) -> float:
        """Calculate EOQ using annual demand and annual holding cost."""
        return math.sqrt(
            (2 * self.params.annual_demand * self.params.ordering_cost)
            / self.params.holding_cost
        )

    def calculate_reorder_point(self, safety_stock: float) -> float:
        """Add safety stock to mean demand during mean lead time."""
        expected_lead_time_demand = (
            self.params.daily_demand * self.params.lead_time_mean
        )
        return expected_lead_time_demand + safety_stock


print("Inventory parameter and policy classes are ready.")

In [ ]:
class PriorityBasedOptimizer(PriorityBasedQRPolicy):
    """Evaluate cost-focused, service-focused, and balanced candidates."""

    def __init__(self, params: InventoryParameters):
        super().__init__(params)
        self.strategies = self._initialize_strategies()

    def _initialize_strategies(self) -> List[Tuple[float, str, Dict]]:
        """Build a min-heap whose negated priorities process larger scores first."""
        strategies = []

        heapq.heappush(
            strategies,
            (-self.criticality_score, 'cost_focused', {}),
        )
        service_priority = -self.params.service_level * 1000
        heapq.heappush(
            strategies,
            (service_priority, 'service_focused', {}),
        )
        heapq.heappush(strategies, (-500, 'balanced', {}))

        return strategies

    def _cost_focused_optimization(self) -> Tuple[float, float, float]:
        """Scale standard safety stock using the stockout-to-holding cost ratio."""
        q = self.calculate_eoq()
        base_ss = self.calculate_safety_stock()
        cost_adjustment = min(
            1.0,
            self.params.stockout_cost / (10 * self.params.holding_cost),
        )
        adjusted_ss = base_ss * cost_adjustment
        r = self.calculate_reorder_point(adjusted_ss)

        return q, r, adjusted_ss

    def _service_focused_optimization(self) -> Tuple[float, float, float]:
        """Add the heuristic service buffer when the target exceeds 0.99."""
        q = self.calculate_eoq()
        safety_stock = self.calculate_safety_stock()

        if self.params.service_level > 0.99:
            safety_stock *= 1.2

        r = self.calculate_reorder_point(safety_stock)
        return q, r, safety_stock

    def _balanced_optimization(self) -> Tuple[float, float, float]:
        """Use EOQ and standard safety stock without strategy adjustments."""
        q = self.calculate_eoq()
        safety_stock = self.calculate_safety_stock()
        r = self.calculate_reorder_point(safety_stock)

        return q, r, safety_stock

    def _calculate_total_cost(
        self,
        q: float,
        r: float,
        safety_stock: float,
    ) -> float:
        """Estimate annual cost using holding, ordering, and stockout components."""
        if q <= 0:
            raise ValueError("Order quantity must be positive.")

        cycle_stock = q / 2
        holding_cost = (
            cycle_stock + safety_stock
        ) * self.params.holding_cost
        ordering_cost = (
            self.params.annual_demand / q
        ) * self.params.ordering_cost

        # This proxy is intentionally simple and is constant across candidates for one item.
        stockout_probability = 1 - self.params.service_level
        expected_stockouts = (
            stockout_probability * self.params.annual_demand / 4
        )
        stockout_cost = expected_stockouts * self.params.stockout_cost

        return holding_cost + ordering_cost + stockout_cost

    def priority_based_optimization(self) -> Dict:
        """Evaluate every queued strategy and retain the least-cost candidate."""
        best_solution = None
        best_cost = float('inf')
        strategy_results = []

        # Rebuild the heap for each run so repeated calls evaluate the same candidates.
        strategies = self._initialize_strategies()

        while strategies:
            priority, strategy_type, _strategy_parameters = heapq.heappop(
                strategies
            )

            if strategy_type == 'cost_focused':
                q, r, safety_stock = self._cost_focused_optimization()
            elif strategy_type == 'service_focused':
                q, r, safety_stock = self._service_focused_optimization()
            else:
                q, r, safety_stock = self._balanced_optimization()

            total_cost = self._calculate_total_cost(q, r, safety_stock)
            result = {
                'Q': q,
                'r': r,
                'safety_stock': safety_stock,
                'total_cost': total_cost,
                'strategy': strategy_type,
                'priority': -priority,
            }
            strategy_results.append(result)

            if total_cost < best_cost:
                best_cost = total_cost
                best_solution = result.copy()

        return {
            'best_solution': best_solution,
            'all_strategies': strategy_results,
            'criticality_score': self.criticality_score,
        }


print("Priority-based candidate evaluation is ready.")

In [ ]:
# Define the CardioStabil example inputs.
cardio_params = InventoryParameters(
    annual_demand=12000,
    lead_time_mean=14,
    lead_time_std=3,
    demand_std=10,
    holding_cost_rate=0.25,
    unit_cost=450,
    ordering_cost=200,
    stockout_cost=2000,
    service_level=0.995,
)

optimizer = PriorityBasedOptimizer(cardio_params)
results = optimizer.priority_based_optimization()

print("=== CRITICALITY ANALYSIS ===")
print(f"Criticality index: {results['criticality_score']:.2f}")

cost_ratio = cardio_params.stockout_cost / cardio_params.holding_cost
demand_cv = cardio_params.demand_std / cardio_params.daily_demand
lead_time_cv = cardio_params.lead_time_std / cardio_params.lead_time_mean

print(f"Stockout-to-holding cost ratio: {cost_ratio:.2f}")
print(f"Daily demand coefficient of variation: {demand_cv:.2f}")
print(f"Lead-time coefficient of variation: {lead_time_cv:.2f}")
print(
    "Criticality calculation: "
    f"{cost_ratio:.2f} × (1 + {demand_cv:.2f}) "
    f"× (1 + {lead_time_cv:.2f}) = {results['criticality_score']:.2f}"
)

print("\n=== STRATEGY COMPARISON ===")
strategy_df = pd.DataFrame(results['all_strategies'])
print(
    strategy_df[
        ['strategy', 'priority', 'Q', 'r', 'safety_stock', 'total_cost']
    ].round(2)
)

best = results['best_solution']
print("\n=== LOWEST-COST EVALUATED CANDIDATE ===")
print(f"Strategy: {best['strategy'].replace('_', ' ').title()}")
print(f"Order quantity (Q): {best['Q']:.2f} units")
print(f"Reorder point (r): {best['r']:.2f} units")
print(f"Safety stock: {best['safety_stock']:.2f} units")
print(f"Estimated annual cost: ${best['total_cost']:,.2f}")

In [ ]:
def create_priority_visualization(results, params):
    """Plot candidate costs, criticality components, queue priorities, and sensitivity."""
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(
        2, 2, figsize=(15, 12)
    )
    fig.suptitle(
        'Priority-Based (Q, r) Policy Analysis — CardioStabil',
        fontsize=16,
        fontweight='bold',
    )

    strategies = results['all_strategies']
    strategy_names = [
        strategy['strategy'].replace('_', ' ').title()
        for strategy in strategies
    ]
    costs = [strategy['total_cost'] for strategy in strategies]

    bars = ax1.bar(
        strategy_names,
        costs,
        color=['#ff6b6b', '#4ecdc4', '#45b7d1'],
    )
    ax1.set_ylabel('Estimated annual cost ($)')
    ax1.set_title('Estimated Cost by Strategy')
    ax1.tick_params(axis='x', rotation=35)
    for bar, cost in zip(bars, costs):
        height = bar.get_height()
        ax1.text(
            bar.get_x() + bar.get_width() / 2,
            height + height * 0.01,
            f'${cost:,.0f}',
            ha='center',
            va='bottom',
            fontweight='bold',
        )

    cost_ratio = params.stockout_cost / params.holding_cost
    demand_cv = params.demand_std / params.daily_demand
    lead_time_cv = params.lead_time_std / params.lead_time_mean
    criticality_components = [
        'Cost ratio',
        'Demand CV',
        'Lead-time CV',
        'Combined index',
    ]
    criticality_values = [
        cost_ratio,
        demand_cv,
        lead_time_cv,
        results['criticality_score'],
    ]
    normalized_values = (
        np.array(criticality_values) / max(criticality_values)
    )
    bars2 = ax2.bar(
        criticality_components,
        normalized_values,
        color=['#e74c3c', '#f39c12', '#3498db', '#9b59b6'],
    )
    ax2.set_ylabel('Value divided by the largest component')
    ax2.set_title('Criticality Index Components')
    ax2.tick_params(axis='x', rotation=35)
    for bar, value in zip(bars2, criticality_values):
        ax2.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.02,
            f'{value:.2f}',
            ha='center',
            va='bottom',
            fontweight='bold',
        )

    for strategy in strategies:
        strategy_name = strategy['strategy'].replace('_', ' ').title()
        ax3.scatter(
            strategy['priority'],
            strategy['Q'],
            s=max(strategy['safety_stock'], 1) * 2,
            alpha=0.6,
            label=strategy_name,
            edgecolors='black',
            linewidth=1,
        )
    ax3.set_xlabel('Priority score (higher scores are processed first)')
    ax3.set_ylabel('Order quantity, Q (units)')
    ax3.set_title('Priority Queue and Candidate Quantities')
    ax3.legend(title='Strategy', bbox_to_anchor=(1.05, 1), loc='upper left')
    ax3.text(
        0.02,
        0.02,
        'Bubble area is proportional to safety stock.',
        transform=ax3.transAxes,
        fontsize=8,
    )
    ax3.grid(True, alpha=0.3)

    cost_ratios = np.linspace(10, 50, 20)
    base_cv_factor = (1 + demand_cv) * (1 + lead_time_cv)
    criticalities = cost_ratios * base_cv_factor
    ax4.plot(cost_ratios, criticalities, color='purple', linewidth=2)
    ax4.scatter(
        [cost_ratio],
        [results['criticality_score']],
        color='red',
        s=100,
        zorder=5,
        label=f'Current ratio: {cost_ratio:.2f}',
    )
    ax4.set_xlabel('Stockout-to-holding cost ratio')
    ax4.set_ylabel('Criticality index')
    ax4.set_title('Criticality Index versus Cost Ratio')
    ax4.legend()
    ax4.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


create_priority_visualization(results, cardio_params)

In [ ]:
def compare_with_baseline(results, params):
    """Compare heuristic candidates with the EOQ and standard-safety-stock baseline."""
    baseline_optimizer = PriorityBasedOptimizer(params)
    baseline_q = baseline_optimizer.calculate_eoq()
    baseline_safety_stock = baseline_optimizer.calculate_safety_stock()
    baseline_r = baseline_optimizer.calculate_reorder_point(
        baseline_safety_stock
    )
    baseline = {
        'Q': baseline_q,
        'r': baseline_r,
        'safety_stock': baseline_safety_stock,
        'strategy': 'Mathematical Formulation',
    }
    baseline_cost = baseline_optimizer._calculate_total_cost(
        baseline['Q'],
        baseline['r'],
        baseline['safety_stock'],
    )
    baseline['total_cost'] = baseline_cost

    comparison_data = results['all_strategies'].copy()
    comparison_data.append(baseline)
    comparison_df = pd.DataFrame(comparison_data)

    best_heuristic = results['best_solution']
    cost_gap = (
        (best_heuristic['total_cost'] - baseline_cost) / baseline_cost * 100
    )

    print("=== PERFORMANCE COMPARISON ===")
    print("\nMathematical-formulation baseline:")
    print(f"   Q: {baseline['Q']:.2f} units")
    print(f"   r: {baseline['r']:.2f} units")
    print(f"   Safety stock: {baseline['safety_stock']:.2f} units")
    print(f"   Estimated annual cost: ${baseline_cost:,.2f}")

    print(
        "\nLowest-cost evaluated heuristic candidate "
        f"({best_heuristic['strategy'].replace('_', ' ').title()}):"
    )
    print(f"   Q: {best_heuristic['Q']:.2f} units")
    print(f"   r: {best_heuristic['r']:.2f} units")
    print(f"   Safety stock: {best_heuristic['safety_stock']:.2f} units")
    print(f"   Estimated annual cost: ${best_heuristic['total_cost']:,.2f}")

    print("\nDifferences from the baseline:")
    print(f"   Estimated-cost gap: {cost_gap:+.2f}%")
    print(
        "   Q difference: "
        f"{best_heuristic['Q'] - baseline['Q']:+.2f} units"
    )
    print(
        "   r difference: "
        f"{best_heuristic['r'] - baseline['r']:+.2f} units"
    )
    print(
        "   Safety-stock difference: "
        f"{best_heuristic['safety_stock'] - baseline['safety_stock']:+.2f} units"
    )

    return comparison_df, cost_gap


comparison_df, cost_gap = compare_with_baseline(results, cardio_params)

print("\n=== FULL COMPARISON TABLE ===")
display_df = comparison_df[
    ['strategy', 'Q', 'r', 'safety_stock', 'total_cost']
].copy()
baseline_cost = comparison_df.loc[
    comparison_df['strategy'] == 'Mathematical Formulation',
    'total_cost',
].iloc[0]
display_df['Cost versus baseline (%)'] = (
    (display_df['total_cost'] - baseline_cost) / baseline_cost * 100
).round(2)
print(display_df.round(2))

In [ ]:
def scalability_analysis():
    """Apply the same three-candidate procedure to the supplied product examples."""
    products = {
        'High_Value_Critical': {
            'annual_demand': 12000,
            'lead_time_mean': 14,
            'lead_time_std': 3,
            'demand_std': 10,
            'holding_cost_rate': 0.25,
            'unit_cost': 450,
            'ordering_cost': 200,
            'stockout_cost': 2000,
            'service_level': 0.995,
        },
        'Medium_Value_Moderate': {
            'annual_demand': 24000,
            'lead_time_mean': 10,
            'lead_time_std': 2,
            'demand_std': 15,
            'holding_cost_rate': 0.20,
            'unit_cost': 120,
            'ordering_cost': 100,
            'stockout_cost': 500,
            'service_level': 0.95,
        },
        'Low_Value_Standard': {
            'annual_demand': 50000,
            'lead_time_mean': 7,
            'lead_time_std': 1,
            'demand_std': 20,
            'holding_cost_rate': 0.15,
            'unit_cost': 25,
            'ordering_cost': 50,
            'stockout_cost': 100,
            'service_level': 0.90,
        },
    }

    product_results = []

    for product_name, params_dict in products.items():
        params = InventoryParameters(**params_dict)
        optimizer = PriorityBasedOptimizer(params)
        result = optimizer.priority_based_optimization()
        best_candidate = result['best_solution']

        product_results.append(
            {
                'Product': product_name.replace('_', ' ').title(),
                'Criticality': result['criticality_score'],
                'Strategy': best_candidate['strategy'].replace('_', ' ').title(),
                'Q': best_candidate['Q'],
                'r': best_candidate['r'],
                'Safety stock': best_candidate['safety_stock'],
                'Estimated annual cost': best_candidate['total_cost'],
                'Unit cost': params.unit_cost,
            }
        )

    results_df = pd.DataFrame(product_results)
    print("=== MULTI-PRODUCT ANALYSIS ===")
    print(results_df.round(2).to_string(index=False))

    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(
        2, 2, figsize=(15, 10)
    )
    fig.suptitle(
        'Multi-Product Heuristic Analysis',
        fontsize=14,
        fontweight='bold',
    )

    ax1.scatter(
        results_df['Criticality'],
        results_df['Estimated annual cost'],
        s=results_df['Unit cost'] / 5,
        alpha=0.7,
        edgecolors='black',
    )
    for _, row in results_df.iterrows():
        ax1.annotate(
            row['Product'],
            (row['Criticality'], row['Estimated annual cost']),
            xytext=(5, 5),
            textcoords='offset points',
            fontsize=8,
        )
    ax1.set_xlabel('Criticality index')
    ax1.set_ylabel('Estimated annual cost ($)')
    ax1.set_title('Criticality versus Estimated Cost')
    ax1.text(
        0.02,
        0.02,
        'Marker area is proportional to unit cost.',
        transform=ax1.transAxes,
        fontsize=8,
    )
    ax1.grid(True, alpha=0.3)

    strategy_counts = results_df['Strategy'].value_counts()
    ax2.pie(
        strategy_counts.values,
        labels=strategy_counts.index,
        autopct='%1.1f%%',
        colors=['#ff6b6b', '#4ecdc4', '#45b7d1'],
    )
    ax2.set_title('Selected Candidate Strategies')

    ax3.scatter(
        results_df['Unit cost'],
        results_df['Q'],
        alpha=0.7,
        s=100,
        edgecolors='black',
    )
    for _, row in results_df.iterrows():
        ax3.annotate(
            row['Product'],
            (row['Unit cost'], row['Q']),
            xytext=(5, 5),
            textcoords='offset points',
            fontsize=8,
        )
    ax3.set_xlabel('Unit cost ($ per unit)')
    ax3.set_ylabel('Order quantity, Q (units)')
    ax3.set_title('Order Quantity versus Unit Cost')
    ax3.grid(True, alpha=0.3)

    ax4.scatter(
        results_df['Criticality'],
        results_df['Safety stock'],
        alpha=0.7,
        s=100,
        edgecolors='black',
    )
    for _, row in results_df.iterrows():
        ax4.annotate(
            row['Product'],
            (row['Criticality'], row['Safety stock']),
            xytext=(5, 5),
            textcoords='offset points',
            fontsize=8,
        )
    ax4.set_xlabel('Criticality index')
    ax4.set_ylabel('Safety stock (units)')
    ax4.set_title('Safety Stock versus Criticality')
    ax4.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    return results_df


scalability_results = scalability_analysis()

In [ ]:
print("=== PRIORITY-BASED HEURISTIC SUMMARY ===")
print("\nALGORITHM CHARACTERISTICS:")
print("   • Three candidate strategies are stored in a priority queue.")
print("   • Each candidate is evaluated before the least-cost candidate is selected.")
print("   • With s strategies, heap operations take O(s log s) time and O(s) space.")
print(f"   • CardioStabil criticality index: {results['criticality_score']:.2f}")
print(
    "   • Lowest-cost evaluated strategy: "
    f"{best['strategy'].replace('_', ' ').title()}"
)

print("\nSELECTED CANDIDATE:")
print(f"   • Order quantity: {best['Q']:.2f} units")
print(f"   • Reorder point: {best['r']:.2f} units")
print(f"   • Safety stock: {best['safety_stock']:.2f} units")
print(f"   • Estimated annual cost: ${best['total_cost']:,.2f}")
print(f"   • Cost gap versus baseline: {cost_gap:+.2f}%")

print("\nINTERPRETATION:")
print(
    "   • The priority score determines evaluation order; estimated cost "
    "determines the selected candidate."
)
print(
    "   • The stockout-cost term is a simplified proxy, so cost comparisons "
    "depend on the stated modeling assumptions."
)
print(
    "   • The selected candidate is the least costly among the strategies "
    "evaluated, not a demonstrated global optimum."
)
print("\nPriority-based candidate evaluation is complete.")